# Step 1: 에이전트 설계 (Math QnA Two-Track Agent)

## 1. 이름
**Trinity RAG Assistant** (트리니티 RAG 어시스턴트)

## 2. 목적
학생의 수학 질문을 받아 학생 상태(수강생 vs 비수강생)에 따라 맞춤형 RAG를 수행합니다. 수강생에게는 강의 타임스탬프와 교재(LaTeX/SVG 메타데이터) 기반의 **프리미엄 해설**을 제공하고, 비수강생에게는 **수평교사경 기출 해설** 기반의 답변을 제공하여 강력한 Q&A 경험 및 수강 유도(Lead Magnet) 퍼널을 구축합니다. 또한 답변 발행 전 선생님(Human)의 검토를 거쳐 답변 품질을 보장합니다.

## 3. 핵심 기능
1. **투트랙 라우팅 (Two-track Routing):** 질문자가 수강생인지 비수강생인지 판별하여 각기 다른 RAG 노드로 라우팅.
2. **맞춤형 RAG 초안 생성 (Contextual Draft Generation):** 
   - *Premium RAG:* 강의 스크립트 + 교재 메타데이터 소스 활용
   - *Public RAG:* 수평교사경 기출 해설 소스 활용
3. **휴먼 인 더 루프 (Teacher Review):** 생성된 AI 초안을 선생님이 검토(Review/Approve/Reject)하고 피드백 반영.

## 4. 그래프 구조 (다이어그램)
```mermaid
graph TD
    START --> Check_Auth_Node
    Check_Auth_Node -- 수강생 --> Premium_RAG_Node
    Check_Auth_Node -- 비수강생 --> Public_RAG_Node
    Premium_RAG_Node --> Teacher_Review_Node
    Public_RAG_Node --> Teacher_Review_Node
    Teacher_Review_Node -- 승인(Approve) --> Publish_Node
    Teacher_Review_Node -- 거절/수정(Reject) --> Revise_Draft_Node
    Revise_Draft_Node --> Teacher_Review_Node
    Publish_Node --> END
```

In [ ]:
import os
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# Step 2: 기초 구축 (LangGraph 구현)

# 1. State 정의
class QnAState(TypedDict):
    question: str
    is_enrolled: bool
    draft_answer: str
    teacher_feedback: str
    final_answer: str
    status: str

# 2. 노드 구현
def check_auth_node(state: QnAState) -> QnAState:
    print(f"[Check Auth] 질문: {state['question']}")
    print(f"-> 수강생 여부: {state['is_enrolled']}")
    return state

def premium_rag_node(state: QnAState) -> QnAState:
    print("[Premium RAG] 강의 타임스탬프 및 교재(LaTeX) 메타데이터 기반 검색 중...")
    draft = f"교재 42p 참고: {state['question']}에 대한 프리미엄 해설입니다."
    return {"draft_answer": draft}

def public_rag_node(state: QnAState) -> QnAState:
    print("[Public RAG] 수평교사경 기출 해설 DB 검색 중...")
    draft = f"기출 변형 해설: {state['question']}에 대한 일반 해설입니다."
    return {"draft_answer": draft}

def teacher_review_node(state: QnAState) -> QnAState:
    print(f"\n[Teacher Review] 현재 초안: {state['draft_answer']}")
    # 데모를 위해 피드백이 없으면 자동 승인되는 것으로 모의(Mock)합니다.
    if state.get("teacher_feedback"):
        print(f"-> 선생님 피드백: {state['teacher_feedback']}")
        return {"status": "rejected"}
    else:
        print("-> 선생님 승인 완료!")
        return {"status": "approved", "final_answer": state['draft_answer']}

def revise_draft_node(state: QnAState) -> QnAState:
    print("[Revise Draft] 피드백을 반영하여 초안 수정 중...")
    revised = state['draft_answer'] + f" \n(추가 반영: {state['teacher_feedback']})"
    # 수정 후에는 피드백을 초기화하여 다시 리뷰 노드에서 승인되도록 함
    return {"draft_answer": revised, "teacher_feedback": ""}

def publish_node(state: QnAState) -> QnAState:
    print(f"\n[Publish] 학생에게 최종 답변 전송:\n{state['final_answer']}")
    return state

# 3. 라우팅 로직 (조건부 엣지)
def route_rag(state: QnAState) -> Literal["premium_rag_node", "public_rag_node"]:
    if state["is_enrolled"]:
        return "premium_rag_node"
    return "public_rag_node"

def route_review(state: QnAState) -> Literal["publish_node", "revise_draft_node"]:
    if state["status"] == "approved":
        return "publish_node"
    return "revise_draft_node"

# 4. 그래프 연결
workflow = StateGraph(QnAState)

workflow.add_node("check_auth_node", check_auth_node)
workflow.add_node("premium_rag_node", premium_rag_node)
workflow.add_node("public_rag_node", public_rag_node)
workflow.add_node("teacher_review_node", teacher_review_node)
workflow.add_node("revise_draft_node", revise_draft_node)
workflow.add_node("publish_node", publish_node)

workflow.set_entry_point("check_auth_node")

workflow.add_conditional_edges(
    "check_auth_node",
    route_rag,
    {
        "premium_rag_node": "premium_rag_node",
        "public_rag_node": "public_rag_node"
    }
)

workflow.add_edge("premium_rag_node", "teacher_review_node")
workflow.add_edge("public_rag_node", "teacher_review_node")

workflow.add_conditional_edges(
    "teacher_review_node",
    route_review,
    {
        "publish_node": "publish_node",
        "revise_draft_node": "revise_draft_node"
    }
)

workflow.add_edge("revise_draft_node", "teacher_review_node")
workflow.add_edge("publish_node", END)

# 컴파일
app = workflow.compile()


In [ ]:
# 5. 실행 테스트

print("=== [테스트 1] 수강생 질문 (프리미엄 RAG) ===")
test_state_1 = {
    "question": "수2 미분계수 정의 파트에서 f'(a)를 어떻게 이해해야 하나요?",
    "is_enrolled": True,
    "teacher_feedback": ""
}
for output in app.stream(test_state_1):
    pass

print("\n=== [테스트 2] 비수강생 질문 (퍼블릭 RAG + 수정 피드백) ===")
test_state_2 = {
    "question": "2024년 9월 모평 22번 문제 해설 부탁드립니다.",
    "is_enrolled": False,
    "teacher_feedback": "학생이 이해하기 쉽게 중간 과정을 좀 더 자세히 써주세요."
}
for output in app.stream(test_state_2):
    pass
